In [2]:
import pyspark
from delta import *

builder = pyspark.sql.SparkSession.builder.appName("Estonia_json_to_delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.driver.memory", "25g") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()


df = spark.read.option("inferSchema", "true").option("multiline", "true").json("data/Estonia.json/ettevotja_rekvisiidid__yldandmed.json")

24/01/17 10:59:17 WARN Utils: Your hostname, long-icttm resolves to a loopback address: 127.0.1.1; using 192.168.1.16 instead (on interface wlp0s20f3)
24/01/17 10:59:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/long/.ivy2/cache
The jars for the packages stored in: /home/long/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-67f6375b-cf40-43ff-b5e9-02a409ca9a75;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/long/Work/adamftd/adam_datalake/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 108ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-67f6375b-cf40-43ff-b5e9-02a409ca9a75
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/4ms)
24/01/17 10:59:18

In [4]:
file_path = "data/Estonia.delta"
df.write.mode("overwrite") \
       .format("delta") \
       .save(file_path)

In [5]:
file_path = "data/Estonia.delta"
df = spark.read.format("delta").load(file_path)
df.show()

+----------------+--------------------+--------------------+
|ariregistri_kood|                nimi|           yldandmed|
+----------------+--------------------+--------------------+
|        16372442|     000 Holdings OÜ|{[{NULL, NULL, NU...|
|        12754230|        001 group OÜ|{[{6543897, 21823...|
|        12652512|   001 Kinnisvara OÜ|{[{10562890, 2307...|
|        16752073|007 Agent & Partn...|{[{10262124, 2363...|
|        11694365|007 Autohaus osaü...|{[{8669966, 30475...|
|        12937781|   013 investment OÜ|{[{7882999, 23110...|
|        14112620|      01Arvutiabi OÜ|{[{6698492, 22837...|
|        10818150|01 Creations Osaü...|{[{10539349, 2113...|
|        16268360|     0207 Stuudio OÜ|{[{2821821, 45356...|
|        16035029|       020 EHITUS OÜ|{[{6432606, 21659...|
|        11728851|              021 OÜ|{[{7248756, 24720...|
|        16240491|            02JDM OÜ|{[{6604266, 23119...|
|        16739552|       044 AI LAB OÜ|{[{9500395, 34099...|
|        16732113|      

In [6]:
df.count()

356325

In [18]:
# This will create a new column for every field in the yldandmed struct
nested_fields = df.select("yldandmed.*").columns
exprs = ["yldandmed.{}".format(field) for field in nested_fields]

# Construct the select expression including the top level fields
select_exprs = ["ariregistri_kood", "nimi"] + exprs

# Select those expressions from the DataFrame
df_flattened = df.select(*select_exprs)

# Show the resulting DataFrame
df_flattened.show(truncate=False)

+----------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------+--------------+-----------------------------+-----------------+-----------------+-------------------+-----------------------+----------------------+--------------------+-------------------+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
df_flattened.count()

356325

In [20]:
output_path = "data/Estonia.transform.delta"
df_flattened.write.format("delta").mode("overwrite").save(output_path)